# TinyViT Training on Google Colab

This notebook runs TinyViT experiments using **Google's free T4 GPU (15GB VRAM)**.

## Setup Overview
1. Code is cloned from **GitHub**
2. Checkpoints are stored in **Google Drive**
3. Training runs on **Colab's GPU**

## First Time Setup
Before running, create this folder structure in your Google Drive:
```
MyDrive/TinyViT/
├── pretrained/     <- Upload pretrained weights here (or download in notebook)
├── checkpoints/    <- Your trained checkpoints go here
└── logits/         <- Saved teacher logits (if using offline distillation)
```

---
## 1. Setup Environment

In [ ]:
#@title 1.1 Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
else:
    print("ERROR: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> Select GPU")

In [ ]:
#@title 1.2 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create directory structure if it doesn't exist
import os
DRIVE_BASE = "/content/drive/MyDrive/TinyViT"
os.makedirs(f"{DRIVE_BASE}/pretrained", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/logits", exist_ok=True)
print(f"Drive mounted. TinyViT folder: {DRIVE_BASE}")

In [ ]:
#@title 1.3 Clone Repository from GitHub
import os

#@markdown Enter your GitHub repository URL:
GITHUB_REPO = "https://github.com/YOUR_USERNAME/TinyViT.git"  #@param {type:"string"}
BRANCH = "ilyas"  #@param {type:"string"}

WORK_DIR = "/content/TinyViT/Cream/TinyViT"

if not os.path.exists("/content/TinyViT"):
    !git clone -b {BRANCH} {GITHUB_REPO} /content/TinyViT
    print("Repository cloned!")
else:
    print("Repository already exists. Pulling latest changes...")
    !cd /content/TinyViT && git pull origin {BRANCH}

%cd {WORK_DIR}
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
#@title 1.4 Install Dependencies
!pip install -q timm yacs
print("Dependencies installed!")

In [ ]:
#@title 1.5 Download CIFAR-100 Dataset
import torchvision
torchvision.datasets.CIFAR100(root='./data', train=True, download=True)
torchvision.datasets.CIFAR100(root='./data', train=False, download=True)
print("CIFAR-100 ready!")

In [ ]:
#@title 1.6 Download/Link Pretrained Weights
import os

DRIVE_PRETRAINED = "/content/drive/MyDrive/TinyViT/pretrained"
LOCAL_PRETRAINED = "./pretrained"

# Create symlink to Drive pretrained folder
if os.path.exists(LOCAL_PRETRAINED):
    !rm -rf {LOCAL_PRETRAINED}
!ln -s {DRIVE_PRETRAINED} {LOCAL_PRETRAINED}

# Download if not already in Drive
if not os.path.exists(f"{DRIVE_PRETRAINED}/tiny_vit_21m_22k_distill.pth"):
    print("Downloading TinyViT-21M pretrained weights...")
    !wget -q -P {DRIVE_PRETRAINED} https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_21m_22k_distill.pth

if not os.path.exists(f"{DRIVE_PRETRAINED}/tiny_vit_5m_22k_distill.pth"):
    print("Downloading TinyViT-5M pretrained weights...")
    !wget -q -P {DRIVE_PRETRAINED} https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_5m_22k_distill.pth

print("\nPretrained weights:")
!ls -lh {LOCAL_PRETRAINED}/

In [ ]:
#@title 1.7 Link Output Directory to Drive (for checkpoint persistence)
import os

DRIVE_OUTPUT = "/content/drive/MyDrive/TinyViT/checkpoints"
LOCAL_OUTPUT = "./output"

# Create symlink so all outputs are saved to Drive
if os.path.exists(LOCAL_OUTPUT) and not os.path.islink(LOCAL_OUTPUT):
    # Move any existing outputs to Drive first
    !mv {LOCAL_OUTPUT}/* {DRIVE_OUTPUT}/ 2>/dev/null || true
    !rm -rf {LOCAL_OUTPUT}

if not os.path.exists(LOCAL_OUTPUT):
    !ln -s {DRIVE_OUTPUT} {LOCAL_OUTPUT}

print(f"Output directory linked to Drive: {DRIVE_OUTPUT}")
print("Your checkpoints will persist even if Colab disconnects!")

---
## 2. Configure & Run Experiment

In [ ]:
#@title 2.1 List Available Configs
!ls -la configs/cifar100/experiments/

In [ ]:
#@title 2.2 Configure Experiment

#@markdown ### Select Config File
CONFIG = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"  #@param {type:"string"}

#@markdown ### Output Directory Name
OUTPUT_NAME = "exp3_teacher_tinyvit21m"  #@param {type:"string"}

#@markdown ### Pretrained Weights (for finetuning)
PRETRAINED = "pretrained/tiny_vit_21m_22k_distill.pth"  #@param {type:"string"}

#@markdown ### Resume Checkpoint (leave empty to start fresh)
RESUME = ""  #@param {type:"string"}

#@markdown ### Teacher Checkpoint (for distillation)
TEACHER_CHECKPOINT = ""  #@param {type:"string"}

#@markdown ### Batch Size (T4 can handle 64-128 for TinyViT-5M, 32-64 for TinyViT-21M)
BATCH_SIZE = 64  #@param {type:"integer"}

#@markdown ### Number of Workers
NUM_WORKERS = 2  #@param {type:"integer"}

# Build paths
OUTPUT_DIR = f"./output/{OUTPUT_NAME}"

# Build override options
opts = [f"DATA.BATCH_SIZE {BATCH_SIZE}", f"DATA.NUM_WORKERS {NUM_WORKERS}"]
if TEACHER_CHECKPOINT:
    opts.append(f"DISTILL.TEACHER_CHECKPOINT {TEACHER_CHECKPOINT}")
OPTS_STR = " ".join(opts)

print(f"Config: {CONFIG}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
#@title 2.3 Run Training
import os

# Set environment variables
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

# Build command
cmd = f"python main.py --cfg {CONFIG} --data-path ./data --output {OUTPUT_DIR}"

if PRETRAINED and os.path.exists(PRETRAINED):
    cmd += f" --pretrained {PRETRAINED}"

if RESUME:
    cmd += f" --resume {RESUME}"

cmd += f" --opts {OPTS_STR}"

print("="*70)
print("RUNNING:")
print(cmd)
print("="*70)

!{cmd}

---
## 3. Quick Experiment Presets

Run one of these cells to configure an experiment, then run cell 2.3

In [ ]:
#@title Preset: Finetune TinyViT-21M Teacher on CIFAR-100
#@markdown This trains the teacher model from ImageNet pretrained weights.
#@markdown Expected accuracy: ~88-91%

CONFIG = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"
OUTPUT_NAME = "teacher_tinyvit21m"
PRETRAINED = "pretrained/tiny_vit_21m_22k_distill.pth"
RESUME = ""
TEACHER_CHECKPOINT = ""
BATCH_SIZE = 64  # T4 can handle this
NUM_WORKERS = 2

OUTPUT_DIR = f"./output/{OUTPUT_NAME}"
OPTS_STR = f"DATA.BATCH_SIZE {BATCH_SIZE} DATA.NUM_WORKERS {NUM_WORKERS}"

print("Configured: Finetune TinyViT-21M Teacher")
print(f"Output will be saved to: {OUTPUT_DIR}")
print("\n>>> Now run cell 2.3 to start training <<<")

In [ ]:
#@title Preset: Online Distillation with Features (21M -> 5M)
#@markdown Trains TinyViT-5M using online distillation from TinyViT-21M teacher.
#@markdown Requires: Trained teacher checkpoint

CONFIG = "configs/cifar100/experiments/exp10_online_distill_tinyvit21m_to_5m.yaml"
OUTPUT_NAME = "student_online_features"
PRETRAINED = ""
RESUME = ""

# Point to your trained teacher
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"

BATCH_SIZE = 32  # Lower batch for online distill (two models in memory)
NUM_WORKERS = 2

OUTPUT_DIR = f"./output/{OUTPUT_NAME}"
OPTS_STR = f"DATA.BATCH_SIZE {BATCH_SIZE} DATA.NUM_WORKERS {NUM_WORKERS} DISTILL.TEACHER_CHECKPOINT {TEACHER_CHECKPOINT}"

print("Configured: Online Distillation with Features")
print(f"Teacher checkpoint: {TEACHER_CHECKPOINT}")
print(f"Output: {OUTPUT_DIR}")

# Check if teacher exists
import os
if os.path.exists(TEACHER_CHECKPOINT):
    print("\n Teacher checkpoint found!")
else:
    print("\n WARNING: Teacher checkpoint not found!")
    print("Train the teacher first using the 'Finetune TinyViT-21M' preset.")

print("\n>>> Now run cell 2.3 to start training <<<")

In [ ]:
#@title Preset: Online Distillation - Logits Only (Baseline)
#@markdown Same as above but without feature distillation.
#@markdown Use this as baseline to measure feature distillation benefit.

CONFIG = "configs/cifar100/experiments/exp10_online_distill_logits_only.yaml"
OUTPUT_NAME = "student_online_logits_only"
PRETRAINED = ""
RESUME = ""
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"
BATCH_SIZE = 32
NUM_WORKERS = 2

OUTPUT_DIR = f"./output/{OUTPUT_NAME}"
OPTS_STR = f"DATA.BATCH_SIZE {BATCH_SIZE} DATA.NUM_WORKERS {NUM_WORKERS} DISTILL.TEACHER_CHECKPOINT {TEACHER_CHECKPOINT}"

print("Configured: Online Distillation (Logits Only)")
print("\n>>> Now run cell 2.3 to start training <<<")

In [ ]:
#@title Preset: Feature Weight Ablation
#@markdown Run ablation on feature loss weight (beta)

BETA = 0.25  #@param [0.1, 0.25, 0.5, 1.0] {type:"raw"}

CONFIG = "configs/cifar100/experiments/exp10_online_distill_tinyvit21m_to_5m.yaml"
OUTPUT_NAME = f"student_beta_{BETA}"
PRETRAINED = ""
RESUME = ""
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"
BATCH_SIZE = 32
NUM_WORKERS = 2

OUTPUT_DIR = f"./output/{OUTPUT_NAME}"
OPTS_STR = f"DATA.BATCH_SIZE {BATCH_SIZE} DATA.NUM_WORKERS {NUM_WORKERS} DISTILL.TEACHER_CHECKPOINT {TEACHER_CHECKPOINT} DISTILL.FEATURE_WEIGHT {BETA}"

print(f"Configured: Feature Weight Ablation (beta={BETA})")
print("\n>>> Now run cell 2.3 to start training <<<")

---
## 4. Utilities

In [ ]:
#@title List All Checkpoints
!find ./output -name "ckpt_best.pth" 2>/dev/null || echo "No checkpoints found"

In [ ]:
#@title Evaluate Checkpoint
import os

EVAL_CONFIG = "configs/cifar100/experiments/exp3_teacher_tinyvit21m.yaml"  #@param {type:"string"}
EVAL_CHECKPOINT = "./output/teacher_tinyvit21m/Exp3-TinyViT21M-Teacher/default/ckpt_best.pth"  #@param {type:"string"}

os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

!python main.py --cfg {EVAL_CONFIG} --data-path ./data --output ./output/eval --resume {EVAL_CHECKPOINT} --eval --opts DATA.NUM_WORKERS 2

In [ ]:
#@title Check Drive Storage
!df -h /content/drive/MyDrive | tail -1

In [ ]:
#@title Monitor GPU Usage
!nvidia-smi